# LangGraph Instrumentation Backends Comparison

This notebook demonstrates using `instrument_graph()` with different observation backends:

- **trace** - Uses the opto trace system
- **trace + otel** - Trace system with OpenTelemetry observation
- **trace + sysmon** - Trace system with Python 3.12+ sys.monitoring
- **trace + otel + sysmon** - Trace with both observers
- **otel** - Pure OpenTelemetry instrumentation  
- **otel + sysmon** - OpenTelemetry with sys.monitoring
- **sysmon** - Pure sys.monitoring (Python 3.12+)

Each backend provides different tracing and profiling capabilities.

In [ ]:
import sys
import time
from langgraph.graph import StateGraph, START, END
from opto.trace.io import instrument_graph

HAS_SYSMON = hasattr(sys, "monitoring")

print(f"Python {sys.version_info.major}.{sys.version_info.minor}")
print(f"sys.monitoring available: {HAS_SYSMON}")

## Define the Graph

Create a simple planner -> synthesizer graph for demonstration.

In [ ]:
def build_graph():
    """Build a simple planner->synth graph."""
    def planner(state):
        return {"plan": f"plan::{state['query']}"}

    def synth(state):
        query = state.get("query", "")
        plan = state.get("plan", "")
        return {"final_answer": f"answer::{query}::{plan}"}

    g = StateGraph(dict)
    g.add_node("planner", planner)
    g.add_node("synth", synth)
    g.add_edge(START, "planner")
    g.add_edge("planner", "synth")
    g.add_edge("synth", END)
    return g

# Test the base graph
test_graph = build_graph()
test_result = test_graph.compile().invoke({"query": "What is CRISPR?"})
print(f"✓ Graph works: {test_result}")

## Test Different Backends

Run the same graph with different instrumentation backends.

In [ ]:
def run_test(name, instrument_kwargs):
    """Run a single instrumentation test."""
    print(f"\n{'='*60}")
    print(f"Test: {name}")
    print(f"{'='*60}")
    try:
        t0 = time.perf_counter()
        
        # Build and instrument graph  
        graph = build_graph()
        if "backend" in instrument_kwargs and instrument_kwargs["backend"] == "trace":
            # For trace backend, pass graph_factory and scope
            instrumented = instrument_graph(
                graph_factory=build_graph,
                scope=globals(),
                **instrument_kwargs
            )
        else:
            # For otel/sysmon, pass compiled graph
            instrumented = instrument_graph(
                graph=graph.compile(),
                **instrument_kwargs
            )
        
        # Invoke
        result = instrumented.invoke({"query": "What is CRISPR?"})
        dt_ms = (time.perf_counter() - t0) * 1000.0
        
        # Extract answer
        answer = result.get("final_answer", result)
        
        print(f"✓ SUCCESS in {dt_ms:.1f}ms")
        print(f"Answer (preview): {str(answer)[:80]}")
        return True
    except Exception as e:
        print(f"✗ FAIL: {e}")
        import traceback
        traceback.print_exc()
        return False

results = {}

### Test 1: Trace Backend Only

In [ ]:
results["trace"] = run_test(
    "backend='trace'",
    {"backend": "trace", "output_key": "final_answer"}
)

### Test 2: Trace + OpenTelemetry

In [ ]:
results["trace+otel"] = run_test(
    "backend='trace', observe_with=('otel',)",
    {
        "backend": "trace",
        "observe_with": ("otel",),
        "output_key": "final_answer"
    }
)

### Test 3: OpenTelemetry Backend Only

In [ ]:
results["otel"] = run_test(
    "backend='otel'",
    {"backend": "otel", "output_key": "final_answer"}
)

### Test 4: sys.monitoring Tests (Python 3.12+)

These tests only run on Python 3.12+ where `sys.monitoring` is available.

In [ ]:
if HAS_SYSMON:
    results["trace+sysmon"] = run_test(
        "backend='trace', observe_with=('sysmon',)",
        {
            "backend": "trace",
            "observe_with": ("sysmon",),
            "output_key": "final_answer"
        }
    )
    
    results["trace+otel+sysmon"] = run_test(
        "backend='trace', observe_with=('otel', 'sysmon')",
        {
            "backend": "trace",
            "observe_with": ("otel", "sysmon"),
            "output_key": "final_answer"
        }
    )
    
    results["otel+sysmon"] = run_test(
        "backend='otel', observe_with=('sysmon',)",
        {
            "backend": "otel",
            "observe_with": ("sysmon",),
            "output_key": "final_answer"
        }
    )
    
    results["sysmon"] = run_test(
        "backend='sysmon'",
        {
            "backend": "sysmon",
            "output_key": "final_answer"
        }
    )
else:
    print("\n⚠️  sys.monitoring tests skipped (requires Python 3.12+)")

## Results Summary

In [ ]:
print("\n" + "="*80)
print("Test Results Summary")
print("="*80)

passed = sum(1 for v in results.values() if v)
total = len(results)

for name, success in results.items():
    status = "✓ PASS" if success else "✗ FAIL"
    print(f"  {name:30s} {status}")

print(f"\nTotal: {passed}/{total} passed")

# Verify critical backends
assert results.get("trace", False), "trace backend should pass"
assert results.get("otel", False), "otel backend should pass"

print("\n✓ All critical tests passed!")
print("="*80)

# LangGraph trace / OTEL / sys.monitoring comparison demo

Compact notebook comparing the new supported configurations:

- `backend="trace"`
- `backend="trace", observe_with=("otel",)`
- `backend="trace", observe_with=("sysmon",)`
- `backend="trace", observe_with=("otel", "sysmon")`
- `backend="otel", observe_with=("sysmon",)`
- `backend="sysmon"`

It prints result previews, observer artifacts, simple structure summaries, and a small timing comparison.

In [ ]:
import sys, time
from langgraph.graph import StateGraph, START, END
from opto.trace import node
from opto.trace.io import instrument_graph, optimize_graph

HAS_SYSMON = hasattr(sys, 'monitoring')

class StubLLM:
    model = 'stub'
    def __call__(self, messages=None, **kwargs):
        class _Msg:
            content = 'stub-response'
        class _Choice:
            message = _Msg()
        class _Resp:
            choices = [_Choice()]
        return _Resp()

def _raw(x):
    return getattr(x, 'data', x)

In [ ]:
planner_prompt = node('Plan: {query}', trainable=True, name='planner_prompt')
synth_prompt = node('Answer: {query} :: {plan}', trainable=True, name='synth_prompt')

def planner_node(state):
    query = _raw(state['query'])
    return {'plan': planner_prompt.data.replace('{query}', str(query))}

def synth_node(state):
    query = _raw(state['query'])
    plan = _raw(state['plan'])
    answer = synth_prompt.data.replace('{query}', str(query)).replace('{plan}', str(plan))
    return {'final_answer': node(answer, name='final_answer_node')}

def build_trace_graph():
    g = StateGraph(dict)
    g.add_node('planner', planner_node)
    g.add_node('synth', synth_node)
    g.add_edge(START, 'planner')
    g.add_edge('planner', 'synth')
    g.add_edge('synth', END)
    return g

def build_plain_graph():
    def planner(state):
        return {'plan': f"plan::{state['query']}"}
    def synth(state):
        return {'final_answer': f"answer::{state['query']}::{state['plan']}"}
    g = StateGraph(dict)
    g.add_node('planner', planner)
    g.add_node('synth', synth)
    g.add_edge(START, 'planner')
    g.add_edge('planner', 'synth')
    g.add_edge('synth', END)
    return g

In [ ]:
def run_case(name, factory):
    t0 = time.perf_counter()
    graph = factory()
    result = graph.invoke({'query': 'What is CRISPR?'})
    dt_ms = (time.perf_counter() - t0) * 1000.0
    answer = result.get('final_answer', result)
    observer_summary = []
    for art in getattr(graph, '_last_observer_artifacts', []):
        if art.carrier == 'sysmon':
            observer_summary.append({'carrier': 'sysmon', 'events': len(art.profile_doc.get('events', []))})
        elif art.carrier == 'otel':
            otlp = art.raw or {}
            spans = otlp.get('resourceSpans', [{}])[0].get('scopeSpans', [{}])[0].get('spans', []) if otlp.get('resourceSpans') else []
            observer_summary.append({'carrier': 'otel', 'spans': len(spans)})
    sysmon_events = len(getattr(graph, '_last_profile_doc', {}).get('events', [])) if getattr(graph, '_last_profile_doc', None) else None
    row = {
        'name': name,
        'answer_preview': str(getattr(answer, 'data', answer))[:80],
        'time_ms': round(dt_ms, 3),
        'observer_summary': observer_summary,
        'sysmon_events': sysmon_events,
    }
    print(row)
    return row

rows = []

rows.append(run_case(
    'trace',
    lambda: instrument_graph(
        backend='trace',
        graph_factory=build_trace_graph,
        scope=globals(),
        graph_agents_functions=['planner_node', 'synth_node'],
        graph_prompts_list=[planner_prompt, synth_prompt],
        output_key='final_answer',
    ),
))

rows.append(run_case(
    'trace+otel',
    lambda: instrument_graph(
        backend='trace',
        observe_with=('otel',),
        graph_factory=build_trace_graph,
        scope=globals(),
        graph_agents_functions=['planner_node', 'synth_node'],
        graph_prompts_list=[planner_prompt, synth_prompt],
        output_key='final_answer',
    ),
))

if HAS_SYSMON:
    rows.append(run_case(
        'trace+sysmon',
        lambda: instrument_graph(
            backend='trace',
            observe_with=('sysmon',),
            graph_factory=build_trace_graph,
            scope=globals(),
            graph_agents_functions=['planner_node', 'synth_node'],
            graph_prompts_list=[planner_prompt, synth_prompt],
            output_key='final_answer',
        ),
    ))
    rows.append(run_case(
        'trace+otel+sysmon',
        lambda: instrument_graph(
            backend='trace',
            observe_with=('otel', 'sysmon'),
            graph_factory=build_trace_graph,
            scope=globals(),
            graph_agents_functions=['planner_node', 'synth_node'],
            graph_prompts_list=[planner_prompt, synth_prompt],
            output_key='final_answer',
        ),
    ))

rows.append(run_case(
    'otel',
    lambda: instrument_graph(
        graph=build_plain_graph(),
        backend='otel',
        llm=StubLLM(),
        initial_templates={'planner_prompt': 'Plan {query}'},
        output_key='final_answer',
    ),
))

if HAS_SYSMON:
    rows.append(run_case(
        'otel+sysmon',
        lambda: instrument_graph(
            graph=build_plain_graph(),
            backend='otel',
            observe_with=('sysmon',),
            llm=StubLLM(),
            initial_templates={'planner_prompt': 'Plan {query}'},
            output_key='final_answer',
        ),
    ))
    rows.append(run_case(
        'sysmon',
        lambda: instrument_graph(
            graph=build_plain_graph(),
            backend='sysmon',
            initial_templates={'planner_prompt': 'Plan {query}'},
            output_key='final_answer',
        ),
    ))

assert any(r['name'] == 'trace' for r in rows)
assert any(r['name'] == 'otel' for r in rows)
if HAS_SYSMON:
    assert any(r['name'] == 'sysmon' for r in rows)

In [ ]:
# baseline-only optimization sanity checks
trace_graph = instrument_graph(
    backend='trace',
    observe_with=('otel',) if not HAS_SYSMON else ('otel', 'sysmon'),
    graph_factory=build_trace_graph,
    scope=globals(),
    graph_agents_functions=['planner_node', 'synth_node'],
    graph_prompts_list=[planner_prompt, synth_prompt],
    output_key='final_answer',
)
trace_opt = optimize_graph(
    trace_graph,
    queries=['What is CRISPR?'],
    iterations=0,
    eval_fn=lambda payload: {
        'score': 1.0 if 'CRISPR' in str(payload['answer']) else 0.0,
        'feedback': 'Keep CRISPR in the final answer.',
    },
)
assert trace_opt.best_iteration == 0
assert trace_opt.best_score == 1.0

if HAS_SYSMON:
    sysmon_graph = instrument_graph(
        graph=build_plain_graph(),
        backend='sysmon',
        initial_templates={'planner_prompt': 'Plan {query}'},
        output_key='final_answer',
    )
    sysmon_opt = optimize_graph(
        sysmon_graph,
        queries=['What is CRISPR?'],
        iterations=0,
        eval_fn=lambda payload: {
            'score': 1.0 if 'CRISPR' in str(payload['answer']) else 0.0,
            'feedback': 'Keep CRISPR in the answer.',
        },
    )
    assert sysmon_opt.best_iteration == 0
    assert sysmon_opt.best_score == 1.0